# 🫀 Heart Disease Prediction — Beginner's Guide to the Full ML Pipeline

**Welcome!** This notebook walks you through a complete machine-learning project step by step.
By the end, you will know how to:

1. Load and understand a real medical dataset
2. Clean and explore the data
3. Train five different ML models
4. Compare their results and pick the best one

---

> **No prior ML experience needed.** Every line of code is explained in plain language.
> Read each explanation cell before running the code cell below it.

---

## 🗂️ What is this dataset?

This is the **UCI Heart Disease Dataset** from the Cleveland Clinic Foundation — one of the most
famous datasets in medical machine learning.

- **1,025 patients** (rows)
- **13 features** (columns) describing each patient's health
- **1 target column** — did the patient have heart disease? (Yes = 1, No = 0)

Our job is to build a model that can **predict heart disease** from those 13 features.


## 📋 Column-by-Column Guide — What Does Each Feature Mean?

Before writing any code, you must understand what the data represents.
Here is every column explained in plain English:

---

### 🔢 Numeric (continuous) features

| Column | Full Name | What it measures | Example |
|--------|-----------|-----------------|---------|
| `age` | Age | Patient's age in years | 54 |
| `trestbps` | Resting Blood Pressure | Blood pressure when sitting still (mm Hg). Healthy adults: ~120. Above 140 is high. | 130 |
| `chol` | Serum Cholesterol | Total cholesterol in the blood (mg/dL). Above 240 is considered high. | 246 |
| `thalach` | Max Heart Rate Achieved | The highest heart rate recorded during an exercise stress test (beats per minute). Higher is generally better — it means the heart can work hard. | 149 |
| `oldpeak` | ST Depression | How much the ECG "ST segment" drops during exercise compared to rest. A larger drop often signals the heart muscle isn't getting enough blood. | 1.07 |

---

### 🏷️ Binary (Yes/No) features — stored as 0 or 1

| Column | Full Name | 0 means | 1 means |
|--------|-----------|---------|---------|
| `sex` | Sex | Female | Male |
| `fbs` | Fasting Blood Sugar > 120 mg/dL | No (normal) | Yes (elevated) |
| `exang` | Exercise-Induced Angina | No chest pain during exercise | Yes, chest pain during exercise |

---

### 🏷️ Categorical features — stored as numbers that represent categories

| Column | Full Name | Values and their meaning |
|--------|-----------|-------------------------|
| `cp` | Chest Pain Type | **0** = Typical Angina (classic heart pain) · **1** = Atypical Angina · **2** = Non-Anginal Pain · **3** = Asymptomatic (no pain) |
| `restecg` | Resting ECG Results | **0** = Normal · **1** = ST-T wave abnormality · **2** = Left ventricular hypertrophy |
| `slope` | Slope of Peak Exercise ST Segment | **0** = Upsloping · **1** = Flat · **2** = Downsloping (downsloping is most concerning) |
| `ca` | Number of Major Vessels Colored by Fluoroscopy | **0–4** — how many of the heart's main blood vessels show blockage on an X-ray dye test. More blocked vessels = worse. |
| `thal` | Thalassemia (Stress Test Blood Flow) | **0** = Normal · **1** = Fixed defect (permanent damage) · **2** = Normal blood flow · **3** = Reversible defect (area gets less blood under stress) |

---

### 🎯 Target column

| Column | Values | Meaning |
|--------|--------|---------|
| `target` | **0** | No heart disease |
| `target` | **1** | Heart disease present |

> ℹ️ The original dataset used values 0–4 for severity.
> We simplified it: **0 = no disease**, **1–4 = disease present** → treated as **1**.


---
## 🧩 Section 1 — Import Libraries

**What is a library?**
Python libraries are pre-built collections of tools. Instead of writing hundreds of lines
of math from scratch, we import them and use them directly.

Here is what each library does:

| Library | Purpose |
|---------|---------|
| `pandas` | Loads and manipulates data in tables (like Excel, but in Python) |
| `numpy` | Fast math on arrays and numbers |
| `sklearn` | The main machine-learning toolkit — contains models, scalers, and metrics |
| `warnings` | We silence unimportant warnings so the output stays clean |

> 💡 **Tip:** You only need to run this cell once. All cells below can then use these tools.


In [ ]:
# ── Import all libraries ──────────────────────────────────────────────────────

import pandas as pd          # data tables
import numpy as np           # numerical operations
from pathlib import Path     # safe file path handling (works on Windows, Mac, Linux)

# ── sklearn: the machine-learning toolkit ─────────────────────────────────────
from sklearn.model_selection import train_test_split      # split data into train / test
from sklearn.preprocessing  import StandardScaler         # normalise features

# ── Five different ML models we will compare ──────────────────────────────────
from sklearn.linear_model   import LogisticRegression
from sklearn.naive_bayes    import GaussianNB
from sklearn.neighbors      import KNeighborsClassifier
from sklearn.svm            import SVC
from sklearn.ensemble       import RandomForestClassifier

# ── Metrics to measure how good each model is ─────────────────────────────────
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

import warnings
warnings.filterwarnings('ignore')   # suppress harmless convergence warnings

print("✅ All libraries imported successfully!")


---
## 📂 Section 2 — Load the Dataset

### Why `skiprows=1`?
The CSV file was saved from Excel, which adds an extra line at the very top:
```
sep=,
```
This is just Excel telling other programs "columns are separated by commas."
It is **not** part of our data. We tell pandas to skip it with `skiprows=1`.

### Why binarise the target?
The original dataset labels heart disease severity as 0, 1, 2, 3, or 4.
For this project we keep it simple:
- **0** → No heart disease
- **1, 2, 3, or 4** → Heart disease present → we convert all to **1**

This turns the problem into a yes/no (binary) question.


In [ ]:
# ── Load the dataset ──────────────────────────────────────────────────────────

print("=" * 60)
print("  UCI Heart Disease Dataset — Cleveland Clinic")
print("  Loading: heart_disease_for_excel.csv")
print("=" * 60)

# skiprows=1 skips the 'sep=,' line that Excel adds at the top
df = pd.read_csv('heart_disease_for_excel.csv', skiprows=1)

# ── Binarise the target column ────────────────────────────────────────────────
# Original values: 0 = no disease, 1/2/3/4 = disease present
# We convert: anything > 0 becomes 1 (True → int → 1)
df['target'] = (df['target'] > 0).astype(int)

print(f"\n✅ Dataset loaded!")
print(f"   Rows    : {df.shape[0]}  (one row = one patient)")
print(f"   Columns : {df.shape[1]}  (13 features + 1 target)")
print(f"\n   Patients WITH heart disease    : {(df['target'] == 1).sum()}  (target = 1)")
print(f"   Patients WITHOUT heart disease : {(df['target'] == 0).sum()}  (target = 0)")
print(f"\nFirst 5 rows of the dataset:")
df.head()


---
## 🔍 Section 3 — Understand the Data (Exploratory Data Analysis)

**Exploratory Data Analysis (EDA)** means getting to know your data before training anything.

We will do five things:

| Step | What we check | Why it matters |
|------|--------------|----------------|
| 3a | Missing values | ML models cannot handle empty cells |
| 3b | Outliers | Extreme values can mislead models |
| 3c | Data types | All features must be numbers |
| 3d | Descriptive statistics | See the typical range for each column |
| 3e | Correlations | Discover which features predict the target best |


### Step 3a — Check for Missing Values

A **missing value** is an empty cell — like a blank in a spreadsheet.
Most ML models will crash or produce wrong results if given missing values.

We check each column and fill any gaps with the **median** of that column.
> **Why median and not mean?**  
> The median is the middle value when sorted. It ignores extreme outliers.
> For example, if most patients have cholesterol ~240 but one entry is 999 (typo),
> the mean would be pulled up, but the median stays close to 240.


In [ ]:
# ── Step 3a: Missing Values ───────────────────────────────────────────────────

print("--- Step 3a: Missing Values ---\n")

missing = df.isnull().sum()   # count NaN cells per column

if missing.sum() == 0:
    print("✅ No missing values found in any column.")
else:
    print("⚠️  Missing values detected:")
    print(missing[missing > 0])
    # Fill gaps with the column median (robust to outliers)
    df.fillna(df.median(numeric_only=True), inplace=True)
    print("\n✅ Missing values filled with column median.")


### Step 3b — Detect Outliers (IQR Method)

An **outlier** is an unusually large or small value — like a cholesterol reading of 564 when most patients are around 240.

We use the **IQR (Interquartile Range)** method:
- Calculate Q1 (25th percentile) and Q3 (75th percentile)
- IQR = Q3 − Q1
- Any value below `Q1 − 1.5 × IQR` or above `Q3 + 1.5 × IQR` is flagged

We only **report** outliers here; we do not remove them because in medical data
an extreme value might be clinically real and informative.


In [ ]:
# ── Step 3b: Outlier Detection ────────────────────────────────────────────────

print("--- Step 3b: Outlier Detection (IQR Method) ---\n")

# We check only the continuous (non-binary) numeric columns
continuous_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

for col in continuous_cols:
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_fence = Q1 - 1.5 * IQR
    upper_fence = Q3 + 1.5 * IQR
    n_out = ((df[col] < lower_fence) | (df[col] > upper_fence)).sum()
    print(f"  {col:<12}: {n_out:>3} outlier(s)   "
          f"[normal range: {lower_fence:.1f} – {upper_fence:.1f}]")

print("\nℹ️  Outliers are NOT removed — in medical data they may be clinically meaningful.")


### Step 3c — Check Data Types

All columns must be numeric (integers or decimals) before we can train a model.
Let's confirm this is the case.


In [ ]:
# ── Step 3c: Data Types ───────────────────────────────────────────────────────

print("--- Step 3c: Data Types ---\n")
print(df.dtypes.to_string())
print("\n✅ All columns are numeric (int64 or float64). No encoding needed.")


### Step 3d — Descriptive Statistics

`df.describe()` gives a quick statistical summary of every column:

| Statistic | Meaning |
|-----------|---------|
| `count` | How many non-missing values |
| `mean` | Average value |
| `std` | Standard deviation — how spread out values are |
| `min / max` | Smallest and largest values |
| `25% / 50% / 75%` | Percentiles (50% = median) |


In [ ]:
# ── Step 3d: Descriptive Statistics ──────────────────────────────────────────

print("--- Step 3d: Descriptive Statistics ---\n")
print(df.describe().round(2).to_string())


### Step 3e — Feature Correlation with Target

**Correlation** measures the linear relationship between a feature and the target (−1 to +1):

| Correlation | Interpretation |
|-------------|----------------|
| Close to **+1** | Feature increases → more likely to have disease |
| Close to **−1** | Feature increases → less likely to have disease |
| Close to **0** | Feature has little linear relationship with disease |

> ⚠️ Correlation does not equal causation! It just shows a statistical relationship.


In [ ]:
# ── Step 3e: Correlation with Target ─────────────────────────────────────────

print("--- Step 3e: Feature Correlation with Target ---\n")

corr = df.corr(numeric_only=True)['target'].drop('target').sort_values(ascending=False)
print(corr.round(4).to_string())

print("\n📌 Key takeaways:")
print("   • exang (exercise-induced angina) and oldpeak (ST depression)")
print("     have the strongest NEGATIVE correlation — both strongly predict disease.")
print("   • cp (chest pain type) and thalach (max heart rate)")
print("     have the strongest POSITIVE correlation with disease.")


---
## ✂️ Section 4 — Train / Test Split

### The fundamental rule of ML: never test on data you trained on

Imagine you give a student the exam answers before the test.
Their score would look perfect but you would learn nothing about whether they understand the material.

The same principle applies to ML:
- We train the model on **80% of patients** (the training set)
- We evaluate it on the remaining **20%** (the test set — data the model has never seen)

### Why stratify?
`stratify=y` ensures the same proportion of sick/healthy patients
appears in both the training and test set. Without it, you might accidentally
put most sick patients in training, making the test set misleadingly easy.

### Why `random_state=42`?
The split involves randomness. Setting a fixed seed (42 is convention) makes
the split identical every time you run the notebook, so your results are **reproducible**.


In [ ]:
# ── Train / Test Split ────────────────────────────────────────────────────────

# X = feature columns (everything except 'target')
# y = the label column (what we want to predict)
X = df.drop('target', axis=1)   # shape: (1025, 13)
y = df['target']                 # shape: (1025,)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,      # 20% for testing
    random_state=42,     # fixed seed → reproducible results
    stratify=y           # same disease ratio in both splits
)

print("=" * 50)
print("  TRAIN / TEST SPLIT")
print("=" * 50)
print(f"  Total patients    : {len(df)}")
print(f"  Training patients : {X_train.shape[0]}  (80%)")
print(f"  Testing  patients : {X_test.shape[0]}  (20%)")
print(f"\n  Training set disease rate : {y_train.mean():.1%}")
print(f"  Testing  set disease rate : {y_test.mean():.1%}  ← same ratio (stratified)")


---
## ⚖️ Section 5 — Feature Scaling

### Why do we need to scale?

Look at the range of our columns:
- `age`: 29 – 77
- `chol`: 126 – 564
- `oldpeak`: 0 – 6.2

Some algorithms (especially **KNN** and **SVM**) measure distances between patients.
If one column has values up to 564 and another only up to 6.2, the large-scale column
will dominate and essentially ignore the small one — that would be unfair and inaccurate.

**StandardScaler** transforms every feature to:
- Mean = 0
- Standard deviation = 1

This puts all features on an equal footing.

### 🚨 The Golden Rule: Fit only on training data

We call `fit_transform` on the training set only, then `transform` (no `fit`) on the test set.

**Why?**  
If we included test data in the `fit` step, the scaler would learn statistics (mean, std)
that include information from the test set. This is called **data leakage** — the model
indirectly "peeks" at test data during training, producing overly optimistic results
that won't generalise to the real world.


In [ ]:
# ── Feature Scaling ───────────────────────────────────────────────────────────

scaler = StandardScaler()

# fit_transform on TRAINING data:
#   - "fit" = learn mean and std of each column FROM training data
#   - "transform" = apply the scaling
X_train_scaled = scaler.fit_transform(X_train)

# transform on TEST data:
#   - use the SAME mean/std learned from training (do NOT refit!)
X_test_scaled = scaler.transform(X_test)

# Check the result: after scaling, mean ≈ 0 and std ≈ 1 for each column
scaled_df = pd.DataFrame(X_train_scaled, columns=X.columns)
print("Mean of each feature after scaling (should be ≈ 0):")
print(scaled_df.mean().round(4).to_string())
print("\nStd  of each feature after scaling (should be ≈ 1):")
print(scaled_df.std().round(4).to_string())
print("\n✅ Scaling applied correctly.")


---
## 📏 Section 6 — How Do We Measure a Model's Performance?

Before training any model, let's understand the four metrics we will use:

### The Confusion Matrix

For a binary classifier, every prediction falls into one of four boxes:

|  | Predicted: No Disease | Predicted: Has Disease |
|--|----------------------|----------------------|
| **Actual: No Disease** | ✅ True Negative (TN) | ❌ False Positive (FP) |
| **Actual: Has Disease** | ❌ False Negative (FN) | ✅ True Positive (TP) |

From these boxes we derive four metrics:

| Metric | Formula | Plain English | When to prioritise |
|--------|---------|--------------|-------------------|
| **Accuracy** | (TP+TN) / Total | % of all predictions that are correct | Balanced datasets |
| **Precision** | TP / (TP+FP) | Of all "disease" predictions, how many were right? | When false alarms are costly |
| **Recall** | TP / (TP+FN) | Of all truly sick patients, how many did we catch? | **Medical diagnosis** — missing a sick patient is dangerous |
| **F1-Score** | 2 × (Precision × Recall) / (Precision + Recall) | Balanced score between Precision and Recall | When both matter |

> 🏥 **In this project, Recall is the most important metric.**  
> A False Negative (telling a sick patient they're healthy) is far worse than a False Positive.

We define a helper function below that we will reuse for every model:


In [ ]:
# ── Evaluation Helper Function ────────────────────────────────────────────────

results = []   # collect results from all models into this list

def evaluate_model(name, y_test, y_pred):
    """
    Print metrics for a trained model and return them as a dict.

    Parameters
    ----------
    name   : str   — model name for display
    y_test : array — true labels from the test set
    y_pred : array — labels predicted by the model
    """
    acc  = accuracy_score (y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score   (y_test, y_pred, zero_division=0)
    f1   = f1_score       (y_test, y_pred, zero_division=0)
    cm   = confusion_matrix(y_test, y_pred)

    print(f"\n{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")
    print(f"  Accuracy  : {acc:.4f}   ({acc*100:.1f}% of all predictions correct)")
    print(f"  Precision : {prec:.4f}   (of 'disease' predictions, {prec*100:.1f}% were right)")
    print(f"  Recall    : {rec:.4f}   ({rec*100:.1f}% of truly sick patients caught)")
    print(f"  F1-Score  : {f1:.4f}")
    print(f"\n  Confusion Matrix:")
    print(f"           Predicted 0   Predicted 1")
    print(f"  Actual 0    {cm[0,0]:>5}         {cm[0,1]:>5}")
    print(f"  Actual 1    {cm[1,0]:>5}         {cm[1,1]:>5}")

    return {
        'Model'    : name,
        'Accuracy' : round(acc,  4),
        'Precision': round(prec, 4),
        'Recall'   : round(rec,  4),
        'F1-Score' : round(f1,   4),
    }

print("✅ evaluate_model() function defined — ready to use.")


---
## 🤖 Section 7 — Baseline Models

We start with simple, well-understood algorithms.
Their scores set a performance **floor** — a stronger model should beat them.

---

### Model 1 — Logistic Regression

Despite its name, Logistic Regression is a **classification** algorithm (not regression).

**How it works:**  
It calculates a weighted sum of all features, then passes the result through a **sigmoid function**
that squashes any number into the range 0–1. That value is interpreted as the probability
of having heart disease.

```
P(disease) = sigmoid( w₁×age + w₂×sex + w₃×cp + … + w₁₃×thal + b )
```

If P > 0.5 → predict disease, else → predict no disease.

The model **learns** the best weights (w₁ … w₁₃) during training.

**Why use it?**
- Simple and fast
- Produces probabilities (not just yes/no)
- Easy to interpret which features matter most


In [ ]:
# ── Model 1: Logistic Regression ──────────────────────────────────────────────

print("Training Logistic Regression...")

lr_model = LogisticRegression(
    random_state=42,   # reproducibility
    max_iter=1000      # allow enough iterations to find the best weights
)

lr_model.fit(X_train_scaled, y_train)     # ← training happens here
lr_preds = lr_model.predict(X_test_scaled)  # ← predict on unseen test data

results.append(evaluate_model('Logistic Regression', y_test, lr_preds))


---
### Model 2 — Naive Bayes (Gaussian)

**Bayes' Theorem** is a formula from probability theory for updating beliefs given evidence.
Naive Bayes applies it to classification.

**The "Naive" assumption:**  
It assumes that all features are **independent** given the class label.
In reality, features like `age` and `thalach` are correlated, so the assumption
is not perfect — but the model still works surprisingly well.

**The "Gaussian" part:**  
It assumes each feature follows a bell-curve (normal/Gaussian) distribution.

**Why use it?**
- Extremely fast to train
- Works well even with small datasets
- A good sanity-check baseline


In [ ]:
# ── Model 2: Naive Bayes ──────────────────────────────────────────────────────

print("Training Naive Bayes...")

nb_model = GaussianNB()
nb_model.fit(X_train_scaled, y_train)
nb_preds = nb_model.predict(X_test_scaled)

results.append(evaluate_model('Naive Bayes', y_test, nb_preds))


---
## 🚀 Section 8 — Advanced Models

---

### Model 3 — K-Nearest Neighbors (KNN, k=7)

**Intuition:** "Show me your neighbours and I will tell you who you are."

**How it works:**  
To classify a new patient, KNN finds the **7 most similar patients** (nearest neighbours)
in the training set and takes a majority vote of their labels.

"Similarity" is measured by **Euclidean distance** — the straight-line distance between
two points in 13-dimensional feature space.

**Why k=7 (odd)?**  
An odd k prevents tie votes. k=7 is a common starting choice for moderately-sized datasets.

**Why is scaling mandatory here?**  
If `chol` ranges 0–564 and `oldpeak` ranges 0–6.2, then `chol` differences dominate the
distance calculation and `oldpeak` becomes irrelevant. Scaling fixes this.


In [ ]:
# ── Model 3: KNN ──────────────────────────────────────────────────────────────

print("Training K-Nearest Neighbors (k=7)...")

knn_model = KNeighborsClassifier(
    n_neighbors=7   # look at the 7 closest patients to decide
)
knn_model.fit(X_train_scaled, y_train)
knn_preds = knn_model.predict(X_test_scaled)

results.append(evaluate_model('KNN (k=7)', y_test, knn_preds))


---
### Model 4 — Support Vector Machine (SVM)

**Intuition:** Draw the widest possible road between two groups.

**How it works:**  
SVM finds a **hyperplane** (a decision boundary in 13D space) that separates
healthy from sick patients while **maximising the margin** — the gap between the
boundary and the nearest patients on each side.

The patients that sit right at the edge of the margin are called **support vectors**
(that's where the name comes from).

**Parameters we use:**
- `kernel='linear'` — assumes the two groups are roughly linearly separable
- `C=1` — a regularisation parameter. Small C → wider margin, allows some misclassifications.
  Large C → tighter fit, may overfit.

**Why is scaling mandatory here?**  
SVM maximises a distance-based margin. Unscaled features distort that distance.


In [ ]:
# ── Model 4: SVM ──────────────────────────────────────────────────────────────

print("Training Support Vector Machine...")

svm_model = SVC(
    kernel='linear',   # linear decision boundary
    C=1,               # regularisation strength
    random_state=42
)
svm_model.fit(X_train_scaled, y_train)
svm_preds = svm_model.predict(X_test_scaled)

results.append(evaluate_model('SVM (Linear)', y_test, svm_preds))


---
### Model 5 — Random Forest (100 Trees)

**Intuition:** Ask 100 different doctors and take a majority vote.

**How it works:**  
Random Forest builds many **Decision Trees**, each trained on:
- A **random subset of patients** (bagging / bootstrap sampling)
- A **random subset of features** at each split

This randomness makes the trees diverse. When predicting, every tree votes,
and the majority class wins. This is called an **ensemble method**.

**Parameters we use:**
- `n_estimators=100` — build 100 trees
- `max_depth=5` — each tree can ask at most 5 yes/no questions (prevents overfitting)

**Advantages:**
- Does not require scaling (trees split on thresholds, not distances)
- Robust to outliers
- Can tell you which features matter most (feature importance)


In [ ]:
# ── Model 5: Random Forest ────────────────────────────────────────────────────

print("Training Random Forest...")

rf_model = RandomForestClassifier(
    n_estimators=100,   # 100 decision trees
    max_depth=5,        # each tree is limited to 5 levels (prevents overfitting)
    random_state=42
)
rf_model.fit(X_train_scaled, y_train)
rf_preds = rf_model.predict(X_test_scaled)

results.append(evaluate_model('Random Forest', y_test, rf_preds))

# ── Bonus: Feature Importance from Random Forest ──────────────────────────────
print("\n--- Feature Importance (which features does Random Forest rely on most?) ---")
importances = pd.Series(rf_model.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=False)
print(importances.round(4).to_string())
print("\n(Higher = more important to the model's decisions)")


---
## 🏆 Section 9 — Final Comparison Report

Now we put all five models side by side and find the best one.

**How to read the table:**
- `Accuracy` — overall correctness
- `Precision` — how trustworthy are the "disease" predictions?
- `Recall` — how many sick patients did we catch? **(most important in medicine)**
- `F1-Score` — balance between precision and recall

> 📌 In a medical context, the model with the **highest Recall** is the safest choice
> because it minimises missed diagnoses.


In [ ]:
# ── Final Comparison Table ────────────────────────────────────────────────────

comparison_df = (
    pd.DataFrame(results)
    .sort_values('Accuracy', ascending=False)
    .reset_index(drop=True)
)

print("\n" + "=" * 72)
print("                    FINAL MODEL COMPARISON REPORT")
print("=" * 72)
print(comparison_df.to_string(index=False))
print("=" * 72)

best_accuracy = comparison_df.iloc[0]
best_recall   = comparison_df.sort_values('Recall', ascending=False).iloc[0]

print(f"\n🥇 Best Accuracy  → {best_accuracy['Model']}"
      f"  ({best_accuracy['Accuracy']*100:.1f}%)")
print(f"🏥 Best Recall    → {best_recall['Model']}"
      f"  ({best_recall['Recall']*100:.1f}%  — catches the most sick patients)")
print(f"\n💡 For clinical deployment, prefer the model with the highest Recall.")


---
## 🎓 Section 10 — What You Learned

Congratulations! You have completed a full, real-world machine learning pipeline.

### Pipeline Summary

| Step | What we did | Key concept |
|------|------------|-------------|
| 1 | Imported libraries | pandas, numpy, sklearn |
| 2 | Loaded and binarised the data | skiprows=1 for Excel files |
| 3 | EDA — missing values, outliers, stats, correlations | Know your data before modelling |
| 4 | Train/test split (80/20, stratified) | Never test on data you trained on |
| 5 | Feature scaling (StandardScaler) | Fit only on training data to avoid leakage |
| 6 | Defined evaluation metrics | Accuracy, Precision, Recall, F1 |
| 7 | Trained 2 baseline models | Logistic Regression, Naive Bayes |
| 8 | Trained 3 advanced models | KNN, SVM, Random Forest |
| 9 | Compared all 5 models | Pick based on the metric that matters for your use case |

---

### What to try next (optional exercises)

1. **Change the test size** — try `test_size=0.30` and see if scores change
2. **Tune KNN** — try `n_neighbors=3`, `5`, `11` and compare results
3. **Random Forest depth** — try `max_depth=3` vs `max_depth=10`
4. **Cross-validation** — instead of one 80/20 split, try `cross_val_score` with 5 folds
5. **ROC Curve** — plot `roc_curve` from sklearn.metrics to visualise the precision/recall trade-off

---

> ✍️ **Remember:** Machine learning is not magic. It is pattern recognition powered by math.
> The better you understand your data and your problem, the better your models will perform.


---
## ▶️ Part 2 — Full Runnable Pipeline

The cells below contain the **complete, executable code** for the entire pipeline.  
Run them **in order from top to bottom** (Shift+Enter on each cell, or Kernel → Restart & Run All).

> Make sure `heart_disease_for_excel.csv` is in the **same folder** as this notebook before running.

| Cell | What it does |
|------|-------------|
| Cell R1 | Import all libraries |
| Cell R2 | Load & binarise the dataset |
| Cell R3 | Preprocessing & EDA (missing values, outliers, stats, correlations) |
| Cell R4 | Train/test split + feature scaling |
| Cell R5 | Define the evaluation helper function |
| Cell R6 | Train baseline models (Logistic Regression, Naive Bayes) |
| Cell R7 | Train advanced models (KNN, SVM, Random Forest) |
| Cell R8 | Print final comparison report |


In [ ]:
# ── R1 · Imports ──

# ╔══════════════════════════════════════════════════════════════╗
# ║         FULL RUNNABLE PIPELINE  — copy-paste & run          ║
# ║  All cells below are self-contained. Run them top-to-bottom.║
# ╚══════════════════════════════════════════════════════════════╝

# ── 1. Imports ────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes  import GaussianNB
from sklearn.neighbors    import KNeighborsClassifier
from sklearn.svm          import SVC
from sklearn.ensemble     import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
)

import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported.")

In [ ]:
# ── R2 · Load Dataset ──

# ── 2. Load Dataset ───────────────────────────────────────────────────────────
print("=" * 60)
print("  UCI Heart Disease Dataset — Cleveland Clinic")
print("  Loading: heart_disease_for_excel.csv")
print("=" * 60)

# skiprows=1 skips the 'sep=,' line Excel adds at the top
df = pd.read_csv('heart_disease_for_excel.csv', skiprows=1)

# Binarise target: 0 = no disease,  1-4 = disease present → 1
df['target'] = (df['target'] > 0).astype(int)

print(f"\n✅ Dataset loaded!")
print(f"   Shape  : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"   Target : 0 = No Disease | 1 = Has Disease")
print(f"\n   WITH    disease : {(df['target'] == 1).sum()}")
print(f"   WITHOUT disease : {(df['target'] == 0).sum()}")
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
# ── R3 · Preprocessing & EDA ──

# ── 3. Data Preprocessing & EDA ──────────────────────────────────────────────
print("=" * 60)
print("  SECTION 3 — PREPROCESSING & EDA")
print("=" * 60)

# ── 3a: Missing Values ────────────────────────────────────────────────────────
print("\n--- 3a: Missing Values ---")
missing = df.isnull().sum()
if missing.sum() == 0:
    print("  ✅ No missing values found.")
else:
    print(missing[missing > 0])
    df.fillna(df.median(numeric_only=True), inplace=True)
    print("  ✅ Filled with column median.")

# ── 3b: Outlier Detection (IQR) ───────────────────────────────────────────────
print("\n--- 3b: Outlier Detection (IQR Method) ---")
for col in ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR    = Q3 - Q1
    n_out  = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    print(f"  {col:<12}: {n_out:>2} outlier(s)  "
          f"[normal range: {Q1-1.5*IQR:.1f} – {Q3+1.5*IQR:.1f}]")

# ── 3c: Feature Encoding ──────────────────────────────────────────────────────
print("\n--- 3c: Feature Encoding ---")
print("  All features are already numeric — no encoding needed.")

# ── 3d: Descriptive Statistics ────────────────────────────────────────────────
print("\n--- 3d: Dataset Statistics ---")
print(df.describe().round(2).to_string())

# ── 3e: Correlation with Target ───────────────────────────────────────────────
print("\n--- 3e: Feature Correlation with Target ---")
corr = df.corr(numeric_only=True)['target'].drop('target').sort_values(ascending=False)
print(corr.round(4).to_string())

In [ ]:
# ── R4 · Split & Scale ──

# ── 4. Train / Test Split (80 / 20, stratified) ──────────────────────────────
print("=" * 60)
print("  SECTION 4 — TRAIN / TEST SPLIT")
print("=" * 60)

X = df.drop('target', axis=1)   # features  (1025 × 13)
y = df['target']                 # labels    (1025,)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,     # 20 % held out for testing
    random_state=42,    # fixed seed → reproducible
    stratify=y,         # same disease ratio in both splits
)

print(f"\n  Total   : {len(df)}")
print(f"  Train   : {X_train.shape[0]}  (80%)")
print(f"  Test    : {X_test.shape[0]}  (20%)")
print(f"  Train disease rate : {y_train.mean():.1%}")
print(f"  Test  disease rate : {y_test.mean():.1%}  ← same (stratified)")

# ── 5. Feature Scaling (StandardScaler) ──────────────────────────────────────
print("\n--- Feature Scaling ---")
scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform on TRAIN only
X_test_scaled  = scaler.transform(X_test)         # transform only (no fit) on TEST
print("  ✅ StandardScaler applied (fit on training data only).")

In [ ]:
# ── R5 · Evaluation Helper ──

# ── 6. Evaluation Helper Function ────────────────────────────────────────────
results = []   # collects metrics from all five models

def evaluate_model(name, y_test, y_pred):
    """Print and return metrics for a trained model."""
    acc  = accuracy_score (y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score   (y_test, y_pred, zero_division=0)
    f1   = f1_score       (y_test, y_pred, zero_division=0)
    cm   = confusion_matrix(y_test, y_pred)

    print(f"\n{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")
    print(f"  Accuracy  : {acc:.4f}   ({acc*100:.1f}% predictions correct)")
    print(f"  Precision : {prec:.4f}   ({prec*100:.1f}% of 'disease' calls were right)")
    print(f"  Recall    : {rec:.4f}   ({rec*100:.1f}% of sick patients caught)  ← key metric")
    print(f"  F1-Score  : {f1:.4f}")
    print(f"\n  Confusion Matrix:")
    print(f"               Predicted 0   Predicted 1")
    print(f"  Actual 0  →  {cm[0,0]:>9}    {cm[0,1]:>9}")
    print(f"  Actual 1  →  {cm[1,0]:>9}    {cm[1,1]:>9}")

    return {'Model': name, 'Accuracy': round(acc,4), 'Precision': round(prec,4),
            'Recall': round(rec,4), 'F1-Score': round(f1,4)}

print("✅ evaluate_model() ready.")

In [ ]:
# ── R6 · Baseline Models ──

# ── 7. Baseline Models ────────────────────────────────────────────────────────
print("=" * 60)
print("  SECTION 7 — BASELINE MODELS")
print("=" * 60)

# ── Model 1: Logistic Regression ──────────────────────────────────────────────
print("\nTraining Logistic Regression…")
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)
results.append(evaluate_model('Logistic Regression', y_test,
                               lr_model.predict(X_test_scaled)))

# ── Model 2: Naive Bayes ──────────────────────────────────────────────────────
print("\nTraining Naive Bayes…")
nb_model = GaussianNB()
nb_model.fit(X_train_scaled, y_train)
results.append(evaluate_model('Naive Bayes', y_test,
                               nb_model.predict(X_test_scaled)))

In [ ]:
# ── R7 · Advanced Models ──

# ── 8. Advanced Models ────────────────────────────────────────────────────────
print("=" * 60)
print("  SECTION 8 — ADVANCED MODELS")
print("=" * 60)

# ── Model 3: KNN ──────────────────────────────────────────────────────────────
print("\nTraining KNN (k=7)…")
knn_model = KNeighborsClassifier(n_neighbors=7)
knn_model.fit(X_train_scaled, y_train)
results.append(evaluate_model('KNN (k=7)', y_test,
                               knn_model.predict(X_test_scaled)))

# ── Model 4: SVM ──────────────────────────────────────────────────────────────
print("\nTraining SVM (linear kernel)…")
svm_model = SVC(kernel='linear', C=1, random_state=42)
svm_model.fit(X_train_scaled, y_train)
results.append(evaluate_model('SVM (Linear)', y_test,
                               svm_model.predict(X_test_scaled)))

# ── Model 5: Random Forest ────────────────────────────────────────────────────
print("\nTraining Random Forest (100 trees, depth=5)…")
rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_model.fit(X_train_scaled, y_train)
results.append(evaluate_model('Random Forest', y_test,
                               rf_model.predict(X_test_scaled)))

# ── Bonus: Feature Importance ─────────────────────────────────────────────────
print("\n--- Feature Importance (Random Forest) ---")
imp = pd.Series(rf_model.feature_importances_, index=X.columns)
print(imp.sort_values(ascending=False).round(4).to_string())

In [ ]:
# ── R8 · Final Report ──

# ── 9. Final Comparative Report ──────────────────────────────────────────────
comparison_df = (
    pd.DataFrame(results)
    .sort_values('Accuracy', ascending=False)
    .reset_index(drop=True)
)

print("\n" + "=" * 72)
print("                    FINAL MODEL COMPARISON REPORT")
print("=" * 72)
print(comparison_df.to_string(index=False))
print("=" * 72)

best_acc    = comparison_df.iloc[0]
best_recall = comparison_df.sort_values('Recall', ascending=False).iloc[0]

print(f"\n🥇 Best Accuracy → {best_acc['Model']}"
      f"  ({best_acc['Accuracy']*100:.1f}%)")
print(f"🏥 Best Recall   → {best_recall['Model']}"
      f"  ({best_recall['Recall']*100:.1f}% of sick patients caught)")
print("\n💡 For clinical use: prioritise Recall to minimise missed diagnoses.")

comparison_df